### File system agent: The model reads and writes files through your code

The model is asked to create `hello.txt`. It can't touch your disk, so it asks **your code** to run `write_file`.

**The key point:** This is the same two-call pattern as tool calling (Supplement 3 in `1-introduction`), with three tools instead of one. The model only writes a request (a function name plus JSON arguments); your Python function does the real work on your machine; a second call shows the model the result so it can confirm.

<pre style="font-size: 0.85em; line-height: 1.35; white-space: pre; overflow-x: auto;">
Call 1     create(messages, tools)
           └─> a request, not an answer -> write_file(filename, content)
Your code  json.loads(arguments) -> call_function -> write_file(...)
           └─> hello.txt is written next to this notebook
           └─> append the request and the result to messages
Call 2     create(messages, tools)
           └─> the confirmation text
</pre>

#### 1. Setup
`load_dotenv()` loads the key from the `.env` file into the environment. `OpenAI(api_key=os.getenv("OPENAI_API_KEY"))` is the long form of `OpenAI()`: Both read the same key, this one just does it in plain sight.

In [1]:
import json
import os
from openai import OpenAI
from dotenv import load_dotenv

In [2]:
load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

#### 2. The three functions (they run on your machine)
Ordinary Python functions built on `os.listdir` and `open`. The model never sees this code.
- Each returns a **dict**, because the result is turned into JSON text and sent back to the model.
- An error is returned as `{"error": ...}` instead of crashing, so the model reads what went wrong (e.g. a missing file) and can tell you.
- The `print` line shows you each time the agent actually acts.
- `encoding="utf-8"` matters on Windows: Without it, `open` uses the Windows default encoding and fails on characters like `é` or emoji.
- `write_file` uses mode `"w"`, which **overwrites** an existing file without asking.

In [3]:
def list_files(directory="."):
    """Lists all files and subdirectories in a given directory."""
    print(f"[Agent Action: Listing files in '{directory}']")
    try:
        files = os.listdir(directory)
        return {"files": files}
    except Exception as e:
        return {"error": str(e)}

In [4]:
def read_file(filename):
    """Reads the entire content of a specified file."""
    print(f"[Agent Action: Reading file '{filename}']")
    try:
        with open(filename, "r", encoding="utf-8") as f:
            content = f.read()
        return {"content": content}
    except Exception as e:
        return {"error": str(e)}

In [5]:
def write_file(filename, content):
    """Writes or overwrites content to a specified file."""
    print(f"[Agent Action: Writing to file '{filename}']")
    try:
        with open(filename, "w", encoding="utf-8") as f:
            f.write(content)
        return {"status": "success", "filename": filename}
    except Exception as e:
        return {"error": str(e)}

#### 3. The tool definitions (what the model reads)
This list is everything the model learns about the three functions:
- `name`: The label the model writes back when it wants that tool.
- `description`: What the model reads to decide which tool fits the request.
- `parameters`: A JSON Schema for the arguments, with their names and types.
- `"required": []` for `list_files`: `directory` is optional, and the Python default `"."` (the current folder) covers it when the model leaves it out.

In [6]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "list_files",
            "description": "List all files and directories in a given directory.",
            "parameters": {
                "type": "object",
                "properties": {
                    "directory": {
                        "type": "string",
                        "description": "The directory to list. Defaults to '.' (current directory).",
                    },
                },
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Read the contents of a specified file.",
            "parameters": {
                "type": "object",
                "properties": {
                    "filename": {
                        "type": "string",
                        "description": "The name (including path) of the file to read.",
                    },
                },
                "required": ["filename"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "write_file",
            "description": "Write or overwrite content to a specified file. Creates the file if it doesn't exist.",
            "parameters": {
                "type": "object",
                "properties": {
                    "filename": {
                        "type": "string",
                        "description": "The name (including path) of the file to write to.",
                    },
                    "content": {
                        "type": "string",
                        "description": "The text content to write into the file.",
                    },
                },
                "required": ["filename", "content"],
            },
        },
    },
]

#### 4. `call_function`: From the name the model writes to the real function
The model writes a function name as a **string**, and you can't call a string. This maps each name to its function. `**args` unpacks the dict into keyword arguments: `{"filename": "hello.txt", ...}` becomes `write_file(filename="hello.txt", ...)`.

In [7]:
def call_function(name, args):
    if name == "list_files":
        return list_files(**args)
    elif name == "read_file":
        return read_file(**args)
    elif name == "write_file":
        return write_file(**args)
    else:
        return {"error": f"Unknown function: {name}"}

#### 5. The conversation
A system message that tells the model it has file tools, and the user's request.

In [8]:
messages = [
    {"role": "system", "content": "You are a helpful coding assistant. You have tools to list, read, and write files on the local filesystem. Always confirm the action taken."},
    {"role": "user", "content": "Please create a file called 'hello.txt' and write 'Hello from my first coding agent!' inside it."},
]

In [9]:
print(messages[1]["content"])

Please create a file called 'hello.txt' and write 'Hello from my first coding agent!' inside it.


#### 6. Call 1: The model asks for a tool
Same `create()` as always, plus `tools=tools`. This call only **decides** which function to run; nothing is written to disk yet. The reply is a request, not an answer:
- `finish_reason` is `'tool_calls'`: The model stopped because it wants a tool run.
- `content` is `None`: There is no answer text yet.
- In `tool_calls`, `arguments` is a **string** of JSON, and `id` pairs this request with the result you send back.

In [10]:
completion = client.chat.completions.create(
    model="gpt-5-nano",
    messages=messages,
    tools=tools,
)

In [11]:
print(type(completion))
print(completion.model_dump_json(indent=2))

<class 'openai.types.chat.chat_completion.ChatCompletion'>
{
  "id": "chatcmpl-EcOlIIrghfcYRxEaUNCHZv13LRFHk",
  "choices": [
    {
      "finish_reason": "tool_calls",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": null,
        "refusal": null,
        "role": "assistant",
        "annotations": [],
        "audio": null,
        "function_call": null,
        "tool_calls": [
          {
            "id": "call_CpyXytielYzf8HkahbtSRmWS",
            "function": {
              "arguments": "{\"filename\":\"hello.txt\",\"content\":\"Hello from my first coding agent!\"}",
              "name": "write_file"
            },
            "type": "function"
          }
        ]
      }
    }
  ],
  "created": 1792820356,
  "model": "gpt-5-nano-2025-08-07",
  "object": "chat.completion",
  "service_tier": "default",
  "system_fingerprint": null,
  "usage": {
    "completion_tokens": 226,
    "prompt_tokens": 288,
    "total_tokens": 514,
    "completion_token

In [12]:
choice = completion.choices[0]

print(choice.finish_reason)
print(choice.message.content)

tool_calls
None


#### 7. Your code runs the tool
This is where `hello.txt` is actually written. It lands next to this notebook, because that is the notebook's working folder.
- If the model answered with plain text instead of a tool request, the cell just prints it.
- Otherwise the model's request (the assistant message) is appended **once, before the loop**. Then each tool call gets its own `"tool"` message with the matching `tool_call_id`.
- `json.loads` turns the arguments string into a dict; `json.dumps` turns the result dict back into text, because a message's `content` must be text.

In [13]:
if not choice.message.tool_calls:
    print("Agent: The model didn't request a tool call. Here's its response:")
    print(choice.message.content)
else:
    # Add the agent's request to call a tool to the message history
    messages.append(choice.message)

    # Execute all tool calls requested
    for tool_call in choice.message.tool_calls:
        print(f"Agent: Needs to call tool '{tool_call.function.name}' with args {tool_call.function.arguments}")
        name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)
        
        # Call the actual Python function
        result = call_function(name, args)
        
        # Add the tool's output to the message history
        messages.append(
            {"role": "tool", "tool_call_id": tool_call.id, "content": json.dumps(result)}
        )

Agent: Needs to call tool 'write_file' with args {"filename":"hello.txt","content":"Hello from my first coding agent!"}
[Agent Action: Writing to file 'hello.txt']


The API remembers nothing between calls, so these 4 messages are the whole conversation the model sees in call 2: The system message, the user message, the model's request, and the tool result. The model's request is an SDK object, not a dict; the SDK converts it to JSON when it sends the request.

In [14]:
messages

[{'role': 'system',
  'content': 'You are a helpful coding assistant. You have tools to list, read, and write files on the local filesystem. Always confirm the action taken.'},
 {'role': 'user',
  'content': "Please create a file called 'hello.txt' and write 'Hello from my first coding agent!' inside it."},
 ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_CpyXytielYzf8HkahbtSRmWS', function=Function(arguments='{"filename":"hello.txt","content":"Hello from my first coding agent!"}', name='write_file'), type='function')]),
 {'role': 'tool',
  'tool_call_id': 'call_CpyXytielYzf8HkahbtSRmWS',
  'content': '{"status": "success", "filename": "hello.txt"}'}]

#### 8. Call 2: The model confirms
All 4 messages and the tools are sent again. The model now sees `{"status": "success", ...}` from `write_file`, so it writes the confirmation. Same model as call 1 (`gpt-5-nano`).

In [15]:
completion_2 = client.chat.completions.create(
    model="gpt-5-nano",
    messages=messages,
    tools=tools,
)

In [16]:
print(completion_2.choices[0].message.content)

All set. I created a file named hello.txt in the current directory and wrote:
Hello from my first coding agent!

Would you like me to display the file contents, or perform any other file operations (append, rename, delete, etc.)?


#### In short
1. **Call 1** sends the request and the three tool definitions. The model replies with a *request* to run `write_file`, not an answer.
2. **Your code** turns the arguments string into a dict, runs the real function (the file is written here), and appends the request plus one `"tool"` message per call.
3. **Call 2** sends everything again, and the model confirms what happened.

**One round only:** This notebook runs the tools once. If call 2 asks for another tool (e.g. `list_files` first, then `read_file`), its `content` is `None` and the notebook stops there. A real agent repeats steps 6–8 in a loop until the model stops asking for tools.